In [1]:
import numpy as np
from pathlib import Path
import warnings

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# TensorFlow for NN surrogate (F8)
import tensorflow as tf
from tensorflow import keras
tf.get_logger().setLevel('ERROR')  # Suppress TF warnings

DATA_DIR = Path("initial_data")

def suggest_ucb_point(
    X,
    y,
    beta=1.5,
    n_candidates=10_000,
    random_state=0,
    bias_point=None,
    bias_scale=0.1,
    bias_fraction=0.5,
    kernel_type="rbf",
    n_restarts=5
):
    """
    Given current data (X, y) for ONE function, suggest the next query point x* using
    a Gaussian Process + Upper Confidence Bound (UCB) heuristic.

    Parameters
    ----------
    X : np.ndarray, shape (N, d)
        The inputs already tried for this function.
        N = number of past points, d = input dimension (2, 3, 4, ..., 8 here).
    y : np.ndarray, shape (N,)
        The outputs observed for those inputs (same order as rows in X).
    beta : float
        Controls exploration vs exploitation in UCB = mean + beta * std.
        - Larger beta = more exploration (try uncertain points).
        - Smaller beta = more exploitation (stay near current best).
    n_candidates : int
        Base number of random candidate points to try in [0, 1]^d.
        We'll possibly increase this for higher dimensions.
    random_state : int
        Seed for the random number generator so results are reproducible.
    bias_point : np.ndarray or None
        If provided, sample some candidates near this point (for exploitation).
    bias_scale : float
        Standard deviation for biased sampling around bias_point.
    bias_fraction : float
        Fraction of candidates to sample near bias_point (rest are uniform).
    kernel_type : str
        "rbf" for smooth functions, "matern" for rougher functions.
    n_restarts : int
        Number of restarts for GP hyperparameter optimization.

    Returns
    -------
    best_x : np.ndarray, shape (d,)
        The chosen next input point (in [0, 1]^d) to submit as query.
    gp : GaussianProcessRegressor
        The fitted GP model, in case you want to inspect predictions later.
    """

    # --------------------------
    # 1. Basic info about X, y
    # --------------------------
    N, d = X.shape

    # --------------------------
    # 2. Build the GP kernel
    # --------------------------
    # WEEK 7 UPDATE: Kernel selection as hyperparameter tuning
    # - RBF: assumes smooth functions (F1, F3, F5, F8)
    # - Matern(nu=2.5): allows more roughness (F2, F4, F6, F7)
    if kernel_type == "matern":
        kernel = 1.0 * Matern(length_scale=np.ones(d), nu=2.5) + WhiteKernel(noise_level=1e-3)
    else:
        kernel = 1.0 * RBF(length_scale=np.ones(d)) + WhiteKernel(noise_level=1e-3)

    # --------------------------
    # 3. Create and fit the GP
    # --------------------------
    # WEEK 7 UPDATE: n_restarts_optimizer=5 for better kernel hyperparameter optimization
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=random_state,
        n_restarts_optimizer=n_restarts,
    )

    gp.fit(X, y)

    # --------------------------
    # 4. Generate candidate points
    # --------------------------
    if d <= 4:
        n = n_candidates
    else:
        n = n_candidates * 2

    rng = np.random.default_rng(random_state)

    if bias_point is not None:
        n_biased = int(n * bias_fraction)
        n_uniform = n - n_biased
        
        Xcand_biased = rng.normal(loc=bias_point, scale=bias_scale, size=(n_biased, d))
        Xcand_biased = np.clip(Xcand_biased, 0, 1)
        
        Xcand_uniform = rng.uniform(0.0, 1.0, size=(n_uniform, d))
        
        Xcand = np.vstack([Xcand_biased, Xcand_uniform])
    else:
        Xcand = rng.uniform(0.0, 1.0, size=(n, d))

    # --------------------------
    # 5. GP predictions at candidates
    # --------------------------
    mu, std = gp.predict(Xcand, return_std=True)

    # --------------------------
    # 6. Compute UCB score
    # --------------------------
    ucb = mu + beta * std

    # --------------------------
    # 7. Mask already-seen points (BUG FIX)
    # --------------------------
    # WEEK 7 BUG FIX: Increased tolerance from 1e-8 to 1e-6 to catch
    # points that are effectively the same but have floating point differences.
    same_point = np.all(
        np.isclose(Xcand[:, None, :], X[None, :, :], atol=1e-6),
        axis=2,
    )
    mask_any_seen = np.any(same_point, axis=1)
    ucb[mask_any_seen] = -np.inf

    # --------------------------
    # 8. Choose the best candidate
    # --------------------------
    best_idx = np.argmax(ucb)
    best_x = Xcand[best_idx]

    return best_x, gp

/opt/anaconda3/envs/tf_env/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
# ======================================
# Neural Network Surrogate (TensorFlow)
# ======================================
# For F8 only (highest dimension), we use a small NN to propose
# candidates via gradient ascent, then let the GP decide via UCB.
#
# WEEK 7 UPDATE: Increased n_random from 1000 to 2000 for wider coverage.

def build_nn_surrogate(input_dim):
    """
    Build a small, regularized MLP surrogate model.
    Small network + dropout to avoid overfitting on tiny data.
    """
    model = keras.Sequential([
        keras.layers.Dense(32, activation='relu', input_shape=(input_dim,)),
        keras.layers.Dropout(0.1),
        keras.layers.Dense(16, activation='relu'),
        keras.layers.Dropout(0.1),
        keras.layers.Dense(1)
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.01),
                  loss='mse')
    return model


def train_nn_surrogate(X, y, epochs=500, verbose=0):
    """
    Train the NN surrogate on observed data.
    Returns the trained model and normalization parameters.
    """
    d = X.shape[1]
    
    y_mean, y_std = y.mean(), y.std() + 1e-8
    y_normalized = (y - y_mean) / y_std
    
    model = build_nn_surrogate(d)
    
    # --------------------------
    # Why Batch Gradient Descent (BGD)?
    # --------------------------
    # We set batch_size=len(X) to use ALL data for each gradient update.
    # This is Batch Gradient Descent, chosen because:
    #   - We have tiny data (~33-43 points), so BGD's cons don't apply
    #   - BGD provides stable, accurate gradient estimates
    #   - SGD would be too noisy with so few points
    model.fit(X, y_normalized, epochs=epochs, batch_size=len(X), verbose=verbose)
    
    return model, y_mean, y_std


def gradient_ascent_on_nn(model, start_point, n_steps=50, lr=0.005, clip_min=0.05, clip_max=0.95):
    """
    Run gradient ascent on the NN surrogate to find a better point.
    Uses TensorFlow's GradientTape to compute gradients of output w.r.t. input.
    
    Conservative settings to avoid extreme boundary queries:
    - lr=0.005 (less aggressive)
    - n_steps=50 (fewer steps)
    - clip to [0.05, 0.95] (avoid boundaries)
    """
    x = tf.Variable(start_point.reshape(1, -1), dtype=tf.float32)
    
    for _ in range(n_steps):
        with tf.GradientTape() as tape:
            y_pred = model(x, training=False)
        
        grad = tape.gradient(y_pred, x)
        x.assign_add(lr * grad)
        x.assign(tf.clip_by_value(x, clip_min, clip_max))
    
    return x.numpy().flatten()


def suggest_nn_gp_hybrid(
    X,
    y,
    beta=1.5,
    n_random=2000,  # WEEK 7: Increased from 1000 to 2000
    n_gradient_steps=50,
    gradient_lr=0.005,
    clip_min=0.05,
    clip_max=0.95,
    top_k=3,
    random_state=0,
    n_restarts=5
):
    """
    NN proposes candidates via gradient ascent, GP selects via UCB.
    
    WEEK 7 UPDATE: 
    - Increased n_random from 1000 to 2000 for wider random coverage
    - Added n_restarts for GP optimization
    - Fixed duplicate point masking
    """
    N, d = X.shape
    rng = np.random.default_rng(random_state)
    
    # Step 1: Train NN surrogate
    tf.random.set_seed(random_state)
    model, y_mean, y_std = train_nn_surrogate(X, y, epochs=500, verbose=0)
    
    # Step 2: Gradient ascent from top-k points
    top_k_indices = np.argsort(y)[-top_k:]
    nn_candidates = []
    
    for idx in top_k_indices:
        start_point = X[idx].copy()
        optimized_point = gradient_ascent_on_nn(
            model, start_point, 
            n_steps=n_gradient_steps, 
            lr=gradient_lr,
            clip_min=clip_min,
            clip_max=clip_max
        )
        nn_candidates.append(optimized_point)
    
    nn_candidates = np.array(nn_candidates)
    
    # Step 3: Add random candidates (safety net)
    random_candidates = rng.uniform(0.0, 1.0, size=(n_random, d))
    
    # Step 4: Combine all candidates
    all_candidates = np.vstack([nn_candidates, random_candidates])
    
    # Step 5: GP scores all candidates via UCB
    # WEEK 7: Using RBF for F8 (high-dimensional but relatively smooth)
    kernel = 1.0 * RBF(length_scale=np.ones(d)) + WhiteKernel(noise_level=1e-3)
    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        random_state=random_state,
        n_restarts_optimizer=n_restarts,
    )
    gp.fit(X, y)
    
    mu, std = gp.predict(all_candidates, return_std=True)
    ucb = mu + beta * std
    
    # WEEK 7 BUG FIX: Increased tolerance to 1e-6
    same_point = np.all(
        np.isclose(all_candidates[:, None, :], X[None, :, :], atol=1e-6),
        axis=2,
    )
    mask_any_seen = np.any(same_point, axis=1)
    ucb[mask_any_seen] = -np.inf
    
    # Step 6: Return best according to GP
    best_idx = np.argmax(ucb)
    best_x = all_candidates[best_idx]
    
    if best_idx < len(nn_candidates):
        source = f"NN gradient ascent (from top-{best_idx + 1} point)"
    else:
        source = "Random candidate"
    
    return best_x, gp, source

In [3]:
# ================================
# Load multi-round inputs/outputs
# ================================

def parse_multiline_lists(filepath):
    """
    Parse a file where each list may span multiple lines.
    """
    with open(filepath) as f:
        content = f.read()
    
    rounds = []
    buffer = ""
    bracket_count = 0
    
    for char in content:
        buffer += char
        if char == '[':
            bracket_count += 1
        elif char == ']':
            bracket_count -= 1
            if bracket_count == 0 and buffer.strip():
                rounds.append(buffer.strip())
                buffer = ""
    
    return rounds

input_lines = parse_multiline_lists("inputs_m18.txt")
output_lines = parse_multiline_lists("outputs_m18.txt")

inputs_rounds = [eval(line, {"np": np, "array": np.array}) for line in input_lines]
outputs_rounds = [eval(line, {"np": np}) for line in output_lines]

n_rounds = len(inputs_rounds)
assert len(outputs_rounds) == n_rounds, "Mismatch between input/output rounds"

for r in range(n_rounds):
    assert len(inputs_rounds[r]) == 8, f"Round {r} inputs != 8"
    assert len(outputs_rounds[r]) == 8, f"Round {r} outputs != 8"

print(f"Loaded data from {n_rounds} previous week(s)")

# ============================
# Week 7 Configuration
# ============================
#
# HYPERPARAMETER TUNING FOCUS:
# This week we treat β, bias, and kernel choice as hyperparameters to tune.
#
# Global changes:
# - n_restarts_optimizer=5 for better GP kernel optimization
# - Matern(2.5) kernel for rough functions (F2, F4, F6, F7)
# - RBF kernel for smoother functions (F1, F3, F5, F8)
#
# Key changes from Week 6:
# - F1: Try edge [0.05, 0.95] (corners and center all ≈0)
# - F2: Keep β=1.5, bias=0.4 + Matern (W6 recovered to 0.529)
# - F3: β=2.0, bias=0.4, Matern (W6's β=1.5/bias=0.5 regressed from -0.013 to -0.042)
# - F4: β=0.75, bias=0.6, Matern (slight regularization against overfitting)
# - F5: Keep β=0.5, bias=0.7, RBF (working beautifully: 1999.95!)
# - F6: β=1.0, bias=0.3, Matern (compromise between 0.2 and 0.4)
# - F7: Keep β=2.0, bias=0.3, Matern + FIX DUPLICATE BUG
# - F8: Keep NN+GP, increase n_random to 2000

beta_map = {
    1: None,   # F1 – manual edge probe
    2: 1.5,    # F2 – keep same, add Matern kernel
    3: 2.0,    # F3 – increase exploration (W6 regression)
    4: 0.75,   # F4 – slight regularization
    5: 0.5,    # F5 – keep heavy exploitation (working!)
    6: 1.0,    # F6 – same β, adjust bias
    7: 2.0,    # F7 – keep exploratory + fix duplicate bug
    8: 1.5,    # F8 – NN+GP hybrid
}

bias_fraction_map = {
    1: None,   # F1 – manual
    2: 0.4,    # F2 – keep same
    3: 0.4,    # F3 – reduce from 0.5 (was too greedy)
    4: 0.6,    # F4 – keep heavy bias
    5: 0.7,    # F5 – keep heavy bias
    6: 0.3,    # F6 – compromise between 0.2 and 0.4
    7: 0.3,    # F7 – keep light bias
    8: None,   # F8 – NN+GP hybrid
}

# Kernel type per function: "matern" for rough, "rbf" for smooth
kernel_map = {
    1: "rbf",      # F1 – doesn't matter (manual)
    2: "matern",   # F2 – noisy, many local peaks
    3: "rbf",      # F3 – narrow optimum, relatively smooth
    4: "matern",   # F4 – rough landscape
    5: "rbf",      # F5 – unimodal, smooth
    6: "matern",   # F6 – noisy
    7: "matern",   # F7 – rough, high-dimensional
    8: "rbf",      # F8 – NN+GP handles complexity
}

print(f"\n{'='*60}")
print(f"WEEK {n_rounds + 1} QUERIES")
print(f"{'='*60}")

for i in range(1, 9):
    # 1. Load initial data
    X_init = np.load(DATA_DIR / f"function_{i}" / "initial_inputs.npy")
    y_init = np.load(DATA_DIR / f"function_{i}" / "initial_outputs.npy")

    X = X_init.copy()
    y = y_init.copy()

    # 2. Append every round's (x, y) for this function
    for r in range(n_rounds):
        x_prev = np.array(inputs_rounds[r][i - 1])
        y_prev = float(outputs_rounds[r][i - 1])

        X = np.vstack([X, x_prev.reshape(1, -1)])
        y = np.append(y, y_prev)

    # WEEK 7 BUG FIX: Verify all previous queries are in X
    print(f"\nFunction {i}: {len(y)} total points ({len(y_init)} initial + {n_rounds} queries)")

    # 3. Find best point so far
    best_idx = np.argmax(y)
    best_point = X[best_idx]
    best_y_so_far = y[best_idx]

    # 4. Generate query based on function-specific strategy
    if i == 1:
        # --------------------------
        # F1: Manual edge search
        # --------------------------
        # W4: [0.05, 0.05] → ≈0
        # W5: [0.95, 0.05] → ≈0
        # W6: [0.50, 0.50] → ≈0
        # Now try vertical edge to continue systematic coverage.
        best_x = np.array([0.05, 0.95])
        strategy = "Manual edge probe [0.05, 0.95]"
    
    elif i == 8:
        # --------------------------
        # F8: NN + GP hybrid
        # --------------------------
        # WEEK 7: Increased n_random from 1000 to 2000 for wider coverage
        best_x, gp, source = suggest_nn_gp_hybrid(
            X, y,
            beta=beta_map[i],
            n_random=2000,  # Increased from 1000
            n_gradient_steps=50,
            gradient_lr=0.005,
            clip_min=0.05,
            clip_max=0.95,
            top_k=3,
            random_state=40 + i,
            n_restarts=5,
        )
        strategy = f"NN+GP hybrid (n_random=2000, {source})"
    
    else:
        # --------------------------
        # F2-F7: GP-UCB with biased sampling
        # --------------------------
        # WEEK 7: Added kernel_type and n_restarts parameters
        best_x, gp = suggest_ucb_point(
            X, y,
            beta=beta_map[i],
            n_candidates=10_000,
            random_state=40 + i,
            bias_point=best_point,
            bias_scale=0.1,
            bias_fraction=bias_fraction_map[i],
            kernel_type=kernel_map[i],
            n_restarts=5,
        )
        strategy = f"GP-UCB (β={beta_map[i]}, bias={bias_fraction_map[i]}, kernel={kernel_map[i]})"

    # 5. Format new query for the portal
    query_str = "-".join(f"{v:.6f}" for v in best_x)

    # WEEK 7 BUG FIX: Check if new query duplicates any existing point
    is_duplicate = np.any(np.all(np.isclose(X, best_x, atol=1e-6), axis=1))
    if is_duplicate:
        print("  ⚠️ WARNING: Suggested query is a duplicate! Check masking logic.")

    # 6. Report
    print(f"  Strategy: {strategy}")
    print(f"  Best y so far: {best_y_so_far:.6f}")
    print(f"  New query to submit: {query_str}")

Loaded data from 6 previous week(s)

WEEK 7 QUERIES

Function 1: 16 total points (10 initial + 6 queries)
  Strategy: Manual edge probe [0.05, 0.95]
  Best y so far: 0.000000
  New query to submit: 0.050000-0.950000

Function 2: 16 total points (10 initial + 6 queries)
  Strategy: GP-UCB (β=1.5, bias=0.4, kernel=matern)
  Best y so far: 0.611205
  New query to submit: 0.715421-0.894940

Function 3: 21 total points (15 initial + 6 queries)
  Strategy: GP-UCB (β=2.0, bias=0.4, kernel=rbf)
  Best y so far: -0.013114
  New query to submit: 0.442070-0.276005-0.343560

Function 4: 36 total points (30 initial + 6 queries)
  Strategy: GP-UCB (β=0.75, bias=0.6, kernel=matern)
  Best y so far: 0.494756
  New query to submit: 0.438063-0.404579-0.417605-0.439886

Function 5: 26 total points (20 initial + 6 queries)
  Strategy: GP-UCB (β=0.5, bias=0.7, kernel=rbf)
  Best y so far: 1999.951927
  New query to submit: 0.000000-1.000000-1.000000-1.000000

Function 6: 26 total points (20 initial + 6 que

/opt/anaconda3/envs/tf_env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


  Strategy: NN+GP hybrid (n_random=2000, Random candidate)
  Best y so far: 9.942989
  New query to submit: 0.295685-0.011031-0.376665-0.068309-0.865947-0.404938-0.099646-0.116629
